# v20 — is decoder cross-attention doing exact retrieval or soft spatial interpolation?

**Question this notebook answers.** Position/station embeddings are added into every
encoder token (K/V) and every decoder query (Q) independently — see `_build_tokens`
(encoder.py) and `StationMAEDecoder.forward` (decoder.py). LayerNorm rescales, it
doesn't erase, so in principle a visible station's query CAN find its own exact key
via the shared positional component surviving through `enc_layers` of self-attention.

This checks whether it actually does, on a **real v20 checkpoint**, for real batches:

- **Part 1** — pull the decoder's cross-attention weights at Δ=0 and measure, for each
  VISIBLE station's query, how much attention mass lands on that station's OWN keys
  vs. is spread across the other N_vis−1 stations. Compares against the uniform
  baseline (1/N_vis) and reports the argmax-key hit rate.
- **Part 2** — the masked-station case: no exact key exists, so check whether mass
  concentrates on the *spatially nearest* visible stations (soft interpolation) or is
  closer to uniform.
- **Part 3** — empirical answer to "what fraction of the post-LayerNorm token is
  content vs. position/station/time?" — measured by projecting the normalised token
  onto the (unit-normalised) content-only and position-only directions, per token,
  rather than assumed from the pre-norm variance decomposition used earlier
  (`token_balance.py`), which does not by itself say what survives normalisation.

No source files are modified. Everything here reads the model/decoder through
public forward calls plus one forward-pre-hook on `cross_attn` to recover attention
weights the production code path discards (`need_weights=False`).


In [3]:
import os, sys
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
PROJ = os.getcwd()
if os.path.join(PROJ, "src") not in sys.path:
    sys.path.insert(0, os.path.join(PROJ, "src"))

import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

torch.set_grad_enabled(False)

# ── User config ──────────────────────────────────────────────────────────
DATA_ROOT  = next(p for p in (
    os.environ.get("DATA_ROOT", ""),
    "/home/renku/work/PeakWeatherDataset",
    os.path.expanduser("~/Documents/ETH/_DAS Project/PeakWeatherDataset"),
    "PeakWeatherDataset",
) if p and os.path.isdir(p))
CHECKPOINT = "checkpoints/full_run_cloud_v20/best.ckpt"
BATCH_SIZE = 8       # number of windows to pool statistics over
N_BATCHES  = 6        # how many batches to average over (bigger = tighter estimate)
DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"


## Load the v20 checkpoint the same way `test.py` does

Reuses `StationMAE.from_cfg` — the single cfg→constructor path — so this notebook can't silently drift onto a different architecture than what was actually trained.

In [ ]:
from model.mae import StationMAE

ckpt = torch.load(CHECKPOINT, map_location=DEVICE, weights_only=False)
saved_cfg = ckpt.get("hyper_parameters", {}).get("cfg", {})

state_dict = {}
for k, v in ckpt["state_dict"].items():
    if not k.startswith("model."):
        continue
    k = k[len("model."):]
    if k.startswith("_orig_mod."):
        k = k[len("_orig_mod."):]
    state_dict[k] = v

model = StationMAE.from_cfg(saved_cfg, dropout=0.0, use_nll_loss=False).to(DEVICE).eval()
missing, unexpected = model.load_state_dict(state_dict, strict=False)
print(f"missing={missing}\nunexpected={unexpected}")
print(f"\nmask_ratio={model.mask_ratio}  residual_head={model.residual_head}  "
      f"query_anchor={model.query_anchor}  cross_attn_decoder={model.decoder.use_cross_attention}")
assert model.decoder.use_cross_attention, "this check is written for the cross-attention decoder"


## Load the test split

Same construction as `notebooks/explore_pipeline.ipynb`, trimmed to what's needed here (no train/val loaders).

In [ ]:
from data.dataset import (
    load_peakweather, build_spatial_features, compute_obs_stats, StationMAEDataset,
)

WINDOW_SIZE        = saved_cfg.get("window", 72)
MAX_DELTA_STEPS    = saved_cfg.get("max_delta", 36)
DELTA_GRID_STRIDE  = saved_cfg.get("delta_grid_stride", 3)

ds = load_peakweather(DATA_ROOT)
spatial, spatial_stats = build_spatial_features(ds)   # (N, 15)
obs_stats = compute_obs_stats(ds, train_years=None)

test_ds = StationMAEDataset(
    ds, window_size=WINDOW_SIZE, delta_steps=MAX_DELTA_STEPS, split="test",
    obs_stats=obs_stats, max_delta_steps=MAX_DELTA_STEPS, cache_dir=DATA_ROOT,
    shared_memory=False, index_mode="blocks",
    delta_mode="fixed_grid", delta_grid_stride=DELTA_GRID_STRIDE,
)
loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
N = spatial.shape[0]
spatial = spatial.to(DEVICE)
print(f"test windows: {len(test_ds)}   stations N={N}   d_model={model.decoder.d_model}")


## Part 1 — visible stations: exact key present, does attention find it?

For a single batch at Δ=0:

1. Run the encoder → `encoded_vis` (B, T·N_vis, d), plus `visible_idx` / `masked_idx`.
2. Build the Δ=0 decoder query exactly as `StationMAEDecoder.forward` does (mask_token
   or anchor + pos/station/time/delta/step embeddings) — done by just calling the real
   `model.decoder(...)` and letting a forward-pre-hook on every layer's `cross_attn`
   record the exact `(query, key, value)` tensors that layer used.
3. Re-invoke each captured `cross_attn` module with `need_weights=True,
   average_attn_weights=False` on those SAME tensors — deterministic in eval mode
   (dropout=0), so this recovers the weights the real forward pass computed internally
   but discarded, without changing a single number the model produced.
4. For each visible station's query row, split attention mass into "own station's
   keys" (all T timesteps of that station) vs. "other stations' keys", and compare to
   the uniform baseline 1/N_vis.


In [ ]:
captured = []  # (layer_idx, module, query, key, value, kwargs)

def _make_hook(layer_idx):
    def hook(module, args, kwargs):
        q = kwargs.get("query", args[0] if len(args) > 0 else None)
        k = kwargs.get("key",   args[1] if len(args) > 1 else None)
        v = kwargs.get("value", args[2] if len(args) > 2 else None)
        captured.append((layer_idx, module, q, k, v, dict(kwargs)))
    return hook

handles = [blk.cross_attn.register_forward_pre_hook(_make_hook(i), with_kwargs=True)
           for i, blk in enumerate(model.decoder.blocks)]

batch = next(iter(loader))
x        = batch["x"].to(DEVICE)
x_mask   = batch["x_mask"].to(DEVICE)
x_hours  = batch["x_hours"].to(DEVICE)
B = x.shape[0]

encoded_vis, masked_idx, visible_idx = model.encoder(x, x_mask, spatial, x_hours)
N_vis = visible_idx.shape[1]
T     = encoded_vis.shape[1] // N_vis     # post-patch time length

anchor = model._query_anchor(encoded_vis, visible_idx, B, N) if model.query_anchor else None
station_masked = torch.zeros(B, N, dtype=torch.bool, device=DEVICE)
station_masked.scatter_(1, masked_idx, True)

y_hours_0     = x_hours[:, -1]                       # Δ=0 target time = last input step
delta_steps_0 = torch.zeros(B, dtype=torch.long, device=DEVICE)

_ = model.decoder(encoded_vis, spatial, y_hours_0, delta_steps_0,
                   station_masked=station_masked, anchor=anchor)
for h in handles:
    h.remove()

print(f"captured {len(captured)} cross-attn calls (one per decoder layer)")
print(f"T={T}  N_vis={N_vis}  N={N}  B={B}")


In [ ]:
# station id owning key-index j:  j = t * N_vis + local_n  ->  visible_idx[b, local_n]
key_station = torch.zeros(B, T * N_vis, dtype=torch.long, device=DEVICE)
local_n_of_key = torch.arange(T * N_vis, device=DEVICE) % N_vis
for b in range(B):
    key_station[b] = visible_idx[b, local_n_of_key]

records = []
for layer_idx, module, q, k, v, kwargs in captured:
    kwargs2 = dict(kwargs)
    kwargs2["need_weights"] = True
    kwargs2["average_attn_weights"] = False
    _, attn_w = module(q, k, v, **kwargs2)          # (B, heads, N, T*N_vis)
    attn_w = attn_w.mean(dim=1)                      # average heads -> (B, N, T*N_vis)

    for b in range(B):
        for local_n, station_n in enumerate(visible_idx[b].tolist()):
            row = attn_w[b, station_n]               # query row for THIS station, all keys
            own_mask   = (key_station[b] == station_n)
            own_mass   = row[own_mask].sum().item()
            argmax_key = row.argmax().item()
            argmax_hit = bool(key_station[b, argmax_key].item() == station_n)
            own_last_step_key = (T - 1) * N_vis + local_n
            own_last_step_mass = row[own_last_step_key].item()
            records.append(dict(layer=layer_idx, b=b, station=station_n,
                                 own_mass=own_mass, argmax_hit=argmax_hit,
                                 own_last_step_mass=own_last_step_mass))

df = pd.DataFrame(records)
baseline = T / (T * N_vis)   # = 1/N_vis: mass a station would get under uniform attention
print(f"uniform baseline (own-station mass if attention were flat): {baseline:.4f}\n")

summary = df.groupby("layer").agg(
    mean_own_mass=("own_mass", "mean"),
    median_own_mass=("own_mass", "median"),
    frac_above_baseline=("own_mass", lambda s: (s > baseline).mean()),
    argmax_hit_rate=("argmax_hit", "mean"),
    mean_own_last_step_mass=("own_last_step_mass", "mean"),
)
summary["own_mass_vs_baseline"] = summary["mean_own_mass"] / baseline
print(summary.round(4))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
df.boxplot(column="own_mass", by="layer", ax=ax)
ax.axhline(baseline, color="red", ls="--", label=f"uniform baseline = {baseline:.3f}")
ax.set_ylabel("attention mass on own station's keys")
ax.set_title("Own-station mass per decoder layer")
ax.legend()
plt.suptitle("")

ax = axes[1]
summary["argmax_hit_rate"].plot(kind="bar", ax=ax, color="steelblue")
ax.axhline(baseline, color="red", ls="--", label=f"chance level ≈ {baseline:.3f}")
ax.set_ylabel("P(argmax key belongs to own station)")
ax.set_title("Exact-retrieval hit rate per layer")
ax.legend()
plt.tight_layout(); plt.show()


**How to read this.** If cross-attention were doing *exact* retrieval for
visible stations, `own_mass` would sit near 1.0 and `argmax_hit_rate` near 1.0 — the
query finds its own T keys and ignores the rest. If it's closer to `baseline`
(1/N_vis), attention isn't privileging the exact match at all despite it being
findable, and whatever the model gets right for visible stations at Δ=0 is coming
from elsewhere (the FFN, the query's own position embedding acting almost like a
bias, or upstream self-attention on the query side). Run this across the
`N_BATCHES` loop below before drawing a conclusion from one batch.


In [ ]:
# Repeat over N_BATCHES batches for a stable estimate.
all_dfs = [df]
loader_iter = iter(loader)
for _ in range(N_BATCHES - 1):
    captured = []
    handles = [blk.cross_attn.register_forward_pre_hook(_make_hook(i), with_kwargs=True)
               for i, blk in enumerate(model.decoder.blocks)]
    try:
        batch = next(loader_iter)
    except StopIteration:
        loader_iter = iter(loader)
        batch = next(loader_iter)
    x, x_mask, x_hours = (batch[k].to(DEVICE) for k in ("x", "x_mask", "x_hours"))
    Bb = x.shape[0]
    encoded_vis, masked_idx, visible_idx = model.encoder(x, x_mask, spatial, x_hours)
    N_vis = visible_idx.shape[1]
    Tb = encoded_vis.shape[1] // N_vis
    anchor = model._query_anchor(encoded_vis, visible_idx, Bb, N) if model.query_anchor else None
    station_masked = torch.zeros(Bb, N, dtype=torch.bool, device=DEVICE)
    station_masked.scatter_(1, masked_idx, True)
    y_hours_0 = x_hours[:, -1]
    delta_steps_0 = torch.zeros(Bb, dtype=torch.long, device=DEVICE)
    _ = model.decoder(encoded_vis, spatial, y_hours_0, delta_steps_0,
                       station_masked=station_masked, anchor=anchor)
    for h in handles:
        h.remove()

    key_station = torch.zeros(Bb, Tb * N_vis, dtype=torch.long, device=DEVICE)
    local_n_of_key = torch.arange(Tb * N_vis, device=DEVICE) % N_vis
    for b in range(Bb):
        key_station[b] = visible_idx[b, local_n_of_key]

    recs = []
    for layer_idx, module, q, k, v, kwargs in captured:
        kwargs2 = dict(kwargs); kwargs2["need_weights"] = True; kwargs2["average_attn_weights"] = False
        _, attn_w = module(q, k, v, **kwargs2)
        attn_w = attn_w.mean(dim=1)
        for b in range(Bb):
            for local_n, station_n in enumerate(visible_idx[b].tolist()):
                row = attn_w[b, station_n]
                own_mask = (key_station[b] == station_n)
                own_mass = row[own_mask].sum().item()
                argmax_hit = bool(key_station[b, row.argmax().item()].item() == station_n)
                recs.append(dict(layer=layer_idx, own_mass=own_mass, argmax_hit=argmax_hit))
    all_dfs.append(pd.DataFrame(recs))

df_all = pd.concat(all_dfs, ignore_index=True)
baseline_all = df_all.groupby("layer").size() * 0 + baseline  # same baseline, N_vis roughly stable
print(f"pooled over {len(all_dfs)} batches, n={len(df_all)} (layer, query) pairs\n")
print(df_all.groupby("layer").agg(
    mean_own_mass=("own_mass", "mean"),
    own_mass_vs_baseline=("own_mass", lambda s: s.mean() / baseline),
    argmax_hit_rate=("argmax_hit", "mean"),
).round(4))


## Part 2 — masked stations: is the fallback nearest-neighbour interpolation?

No exact key exists for a masked station. If attention is doing something sensible
with the leftover positional signal, mass should concentrate on the geographically
*closest* visible stations rather than being flat or effectively random.


In [ ]:
easting_northing = spatial[:, :2].cpu()   # normalised, but distances are monotonic in the raw scale

def nearest_rank(distances, candidate_idx):
    order = np.argsort(distances)
    return int(np.where(order == candidate_idx)[0][0])   # 0 = nearest

# Reuse the LAST captured batch (still in scope) for the masked-station analysis.
records_m = []
for layer_idx, module, q, k, v, kwargs in captured:
    kwargs2 = dict(kwargs); kwargs2["need_weights"] = True; kwargs2["average_attn_weights"] = False
    _, attn_w = module(q, k, v, **kwargs2)
    attn_w = attn_w.mean(dim=1)                 # (B, N, T*N_vis)
    for b in range(Bb):
        vis_stations = visible_idx[b].tolist()
        for station_n in masked_idx[b].tolist():
            row = attn_w[b, station_n]
            # mass per visible station = sum over its T keys
            mass_per_station = row.view(Tb, N_vis).sum(dim=0).cpu().numpy()  # (N_vis,)
            d = np.linalg.norm(
                easting_northing.numpy()[vis_stations] - easting_northing.numpy()[station_n], axis=1)
            nearest_local = int(np.argmin(d))
            top_mass_local = int(np.argmax(mass_per_station))
            # correlation between "closeness rank" and "attention mass rank" across the N_vis candidates
            rank_dist = np.argsort(np.argsort(d))
            rank_mass = np.argsort(np.argsort(-mass_per_station))
            rho = np.corrcoef(rank_dist, rank_mass)[0, 1]
            records_m.append(dict(layer=layer_idx, station=station_n,
                                   nearest_gets_top_mass=(nearest_local == top_mass_local),
                                   spearman_dist_vs_mass=rho,
                                   mass_on_nearest=float(mass_per_station[nearest_local])))

dfm = pd.DataFrame(records_m)
print(dfm.groupby("layer").agg(
    frac_nearest_gets_top_mass=("nearest_gets_top_mass", "mean"),
    mean_spearman_dist_vs_mass=("spearman_dist_vs_mass", "mean"),   # negative = closer -> more mass, as expected
    mean_mass_on_nearest=("mass_on_nearest", "mean"),
).round(4))
print(f"\nuniform baseline mass on ANY single station (incl. nearest) = {baseline:.4f}")
print("spearman_dist_vs_mass near 0 -> no spatial structure; strongly negative -> nearer stations get more mass")


## Part 3 — how much of the post-LayerNorm token is content?

`token_balance.py`'s Var(obs)/Var(total) figure (0.04% → 23.6% across v15→v20) is
computed on the **pre-norm sum** `var_tokens + pos_e + station_e + temp_emb + step_e`.
`token_norm` (a LayerNorm) then rescales each token to unit variance and applies a
learned elementwise affine (`γ, β`) — a single global rescaling per feature dimension,
not a per-component one, so the pre-norm variance share doesn't directly say what
survives.

What's actually measurable: project the POST-norm token onto the unit direction of
its own content-only component (`var_tokens`) and separately onto its own
position-bundle-only component (`pos_e + station_e`), per token. The squared
projection is the fraction of that token's (unit) energy pointing along each
direction — a proper post-normalisation content share, not inferred from pre-norm
variance.


In [ ]:
enc = model.encoder
B2 = x.shape[0]
x_flat    = x.view(B2 * x.shape[1], N, x.shape[-1])
mask_flat = x_mask.view(B2 * x.shape[1], N, x.shape[-1])
_static   = spatial.unsqueeze(0).expand(B2, -1, -1) if enc.static_in_token else None
if _static is not None:
    _static = _static.unsqueeze(1).expand(-1, x.shape[1], -1, -1).reshape(B2 * x.shape[1], N, -1)

with torch.no_grad():
    var_tokens = enc.var_proj(x_flat, mask_flat, static=_static)      # (B*W, N, d)
    var_tokens = var_tokens.view(B2, x.shape[1], N, enc.d_model)

    sp = spatial.unsqueeze(0) if spatial.dim() == 2 else spatial
    pos_e     = enc.pos_emb(sp[..., :2]).unsqueeze(1)                 # (1, 1, N, d)
    station_e = enc.station_emb(sp[..., 2:]).unsqueeze(1)             # (1, 1, N, d)
    temp_emb  = enc.temporal_emb(x_hours).unsqueeze(2)                # (B, W, 1, d)
    step_idx  = torch.arange(x.shape[1], device=DEVICE)
    step_e    = enc.step_emb(step_idx).view(1, x.shape[1], 1, enc.d_model)

    position_bundle = pos_e + station_e                               # (1, 1, N, d), broadcast
    full_token = var_tokens + position_bundle + temp_emb + step_e
    normed = enc.token_norm(full_token)                                # (B, W, N, d) — POST-norm token

    content_dir  = F.normalize(var_tokens, dim=-1)
    position_dir = F.normalize(position_bundle.expand_as(var_tokens), dim=-1)
    normed_unit  = F.normalize(normed, dim=-1)

    content_energy  = (normed_unit * content_dir).sum(-1).pow(2)     # (B, W, N) in [0,1]
    position_energy = (normed_unit * position_dir).sum(-1).pow(2)

print("Post-LayerNorm energy fraction along each component's own direction (mean ± std over B,W,N):")
print(f"  content  (var_proj)         : {content_energy.mean().item():.4f} ± {content_energy.std().item():.4f}")
print(f"  position (pos_emb+station_emb): {position_energy.mean().item():.4f} ± {position_energy.std().item():.4f}")
print(f"  (these are NOT complementary — token, content and position directions are not orthogonal,"
      f" so they don't have to sum to 1; both numbers being large is possible and expected.)")


**Caveat on Part 3.** This measures projection onto each RAW component's own
direction, not a true orthogonal decomposition (var_tokens, position_bundle, temp_emb
and step_e are not mutually orthogonal vectors — they're just added). It's the
honest empirical analogue of the pre-norm `token_balance.py` number, adapted to
survive normalisation, not a claim that content and position partition the token's
variance cleanly. Read it as "how much does content still look like itself after
LayerNorm" rather than a percentage that has to add up to 100.
